# Clase 5 · E07: lectura, transformación y ejecución con Spark
Viernes 16 de octubre de 2026: demostración docente; sábado 17: ejecución propia y entrega en clase. Preparado por el PhD Esteban Hernández, CyberColombia.

[Taller E07](../../talleres/E07.md) · [WSL](../../docs/instalacion-wsl.md) · [Linux/macOS](../../docs/instalacion-linux-macos.md)

**Antes del receso:** datos, tipos, Parquet, sesión Spark, transformaciones y acciones.
**Después del receso:** agregación, plan, materialización, equivalencia DuckDB y evidencias.

## Preparación
Usar el `.venv` de la raíz y JDK 21 según la guía del sistema. Desde la raíz del repositorio, con el entorno activo:
```bash
python -m pip install -r kit/requirements_spark.txt
java -version
cd kit
python 00_datos.py --descargar eva divipola
python 00_datos.py --verificar eva divipola
python -m jupyter lab
```
Seleccionar el kernel del mismo entorno. Si `JAVA_HOME` no está configurado, aplicar la guía antes de abrir Jupyter; reiniciar el servidor para heredar las variables. Trabajar en `local[2]`, incluso con 8 GB de RAM.

Este notebook prepara sus propios productos desde los CSV originales, sin ejecutar `talleres.py`. Las salidas están en `kit/salidas/E07`; las escrituras reemplazan únicamente los productos homónimos de este ejercicio. Los originales no se modifican.

In [1]:
from pathlib import Path
import os, sys, json
import pandas as pd
import duckdb
from IPython.display import display
repositorio = Path("/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate")
# Linux/macOS: repositorio = Path.home() / "bigdata/big-data-postgraduate"
KIT = repositorio / "kit"
RAW = KIT / "data/raw"
OUT = KIT / "salidas/E07"
assert KIT.is_dir(), "Ajusta repositorio a tu clon local"
OUT.mkdir(parents=True, exist_ok=True)
# Los workers usan el mismo Python del kernel y su .venv.
os.environ["PYSPARK_PYTHON"] = sys.executable
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("E07-aprendizaje")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC").getOrCreate())
print("Python:", sys.executable, "Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Shuffle:", spark.conf.get("spark.sql.shuffle.partitions"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/17 16:36:58 WARN Utils: Your hostname, POS-316-A396596, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/17 16:36:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/17 16:37:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Python: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python Spark: 4.0.1
Master: local[2]
Shuffle: 4


## 1. Leer como texto e inspeccionar
Conservar códigos con ceros iniciales. Primero ver columnas, valores y tipos; luego elegir los campos. La preparación pandas es local y se mide separada del procesamiento Spark.

In [2]:
eva_texto = pd.read_csv(RAW / "eva.csv", dtype="string", keep_default_na=False)
divipola = pd.read_csv(RAW / "divipola.csv", dtype="string", keep_default_na=False)
display(eva_texto.head())
display(eva_texto.dtypes)
display(divipola.head())
print("Filas originales:", len(eva_texto), "DIVIPOLA:", len(divipola))

,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Grupo cultivo,Subgrupo,Cultivo,Desagregación cultivo,Año,Periodo,Área sembrada,Área cosechada,Producción,Rendimiento,Ciclo del cultivo,Estado físico del cultivo,Código del cultivo,Nombre científico del cultivo
0,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021A,8.00,8.00,128.00,16.00,Transitorio,En fresco,1050600,Apium graveolens
1,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021B,8.00,8.00,152.00,19.00,Transitorio,En fresco,1050600,Apium graveolens
2,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2022,2022A,8.00,8.00,127.44,15.93,Transitorio,En fresco,1050600,Apium graveolens
3,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2022,2022B,8.00,8.00,152.00,19.00,Transitorio,En fresco,1050600,Apium graveolens
4,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2023,2023A,8.00,7.00,133.00,19.00,Transitorio,En fresco,1050600,Apium graveolens


Código Dane departamento         string[python]
Departamento                     string[python]
Código Dane municipio            string[python]
Municipio                        string[python]
Grupo cultivo                    string[python]
Subgrupo                         string[python]
Cultivo                          string[python]
Desagregación cultivo            string[python]
Año                              string[python]
Periodo                          string[python]
Área sembrada                    string[python]
Área cosechada                   string[python]
Producción                       string[python]
Rendimiento                      string[python]
Ciclo del cultivo                string[python]
Estado físico del cultivo        string[python]
Código del cultivo               string[python]
Nombre científico del cultivo    string[python]
dtype: object

,Código Departamento,Nombre Departamento,Código Municipio,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada,longitud,Latitud
0,05,ANTIOQUIA,05001,MEDELLÍN,Municipio,"-75,581775","6,246631"
1,05,ANTIOQUIA,05002,ABEJORRAL,Municipio,"-75,428739","5,789315"
2,05,ANTIOQUIA,05004,ABRIAQUÍ,Municipio,"-76,064304","6,632282"
3,05,ANTIOQUIA,05021,ALEJANDRÍA,Municipio,"-75,141346","6,376061"
4,05,ANTIOQUIA,05030,AMAGÁ,Municipio,"-75,702188","6,038708"


Filas originales: 166732 DIVIPOLA: 1122


## 2. Convertir y conservar los problemas
Renombrar, convertir con `to_numeric` y marcar el dominio del cociente: área positiva, producción no negativa y año interpretable. No calcular un promedio simple de rendimientos. Las filas fuera del dominio se guardan para revisión con su número de fila; no se borran del producto EVA.

In [3]:
renombres = {"Código Dane municipio": "codigo_municipio", "Código Dane departamento": "codigo_departamento",
    "Año": "anio", "Cultivo": "cultivo", "Periodo": "periodo", "Estado físico del cultivo": "estado_fisico",
    "Área cosechada": "area_ha", "Producción": "produccion_t", "Rendimiento": "rendimiento_publicado"}
eva = eva_texto[list(renombres)].rename(columns=renombres).copy()
eva["fila_fuente"] = range(1, len(eva) + 1)
for campo, ancho in [("codigo_municipio", 5), ("codigo_departamento", 2)]:
    eva[campo] = eva[campo].str.strip().str.zfill(ancho)
for campo in ["area_ha", "produccion_t", "rendimiento_publicado", "anio"]:
    eva[campo + "_texto"] = eva[campo]
    eva[campo] = pd.to_numeric(eva[campo], errors="coerce")
    errores = eva[campo].isna() & eva[campo + "_texto"].str.strip().ne("")
    print(campo, "no convertibles:", int(errores.sum()))
    display(eva.loc[errores, ["fila_fuente", campo + "_texto"]].head())
# Tipos estables en Parquet, compatibles con Spark y DuckDB.
assert eva["anio"].dropna().mod(1).eq(0).all(), "Revisar años fraccionarios"
eva["anio"] = eva["anio"].astype("Int64")
for campo in ["area_ha", "produccion_t", "rendimiento_publicado"]:
    eva[campo] = eva[campo].astype("Float64")
eva["apto_rendimiento"] = (eva["area_ha"].gt(0) & eva["produccion_t"].ge(0) & eva["anio"].notna()).fillna(False)
revision = eva.loc[~eva["apto_rendimiento"]].copy()
revision.to_csv(OUT / "eva_revision.csv", index=False)
assert len(eva) == int(eva["apto_rendimiento"].sum()) + len(revision)
assert int(eva["apto_rendimiento"].sum()) == 159616, "Control del corte docente; revisar fuente si cambió"
display(revision.head())

area_ha no convertibles: 0


,fila_fuente,area_ha_texto


produccion_t no convertibles: 0


,fila_fuente,produccion_t_texto


rendimiento_publicado no convertibles: 0


,fila_fuente,rendimiento_publicado_texto


anio no convertibles: 0


,fila_fuente,anio_texto


,codigo_municipio,codigo_departamento,anio,cultivo,periodo,estado_fisico,area_ha,produccion_t,rendimiento_publicado,fila_fuente,area_ha_texto,produccion_t_texto,rendimiento_publicado_texto,anio_texto,apto_rendimiento
201,05001,05,2023,Papa,2023A,En fresco,0.0,0.0,0.0,202,0.00,0.00,0.00,2023,False
304,05001,05,2022,Yuca,2022A,En fresco,0.0,0.0,0.0,305,0.00,0.00,0.00,2022,False
478,05004,05,2023,Cebolla de bulbo,2023A,En fresco,0.0,0.0,0.0,479,0.00,0.00,0.00,2023,False
525,05004,05,2020,Gulupa o cholupa,2020,En fresco,0.0,0.0,0.0,526,0.00,0.00,0.00,2020,False
621,05004,05,2024,Plátano,2024,En fresco,0.0,0.0,0.0,622,0.00,0.00,0.00,2024,False


## 3. Escribir y volver a leer los Parquet
Preparar una dimensión única por código para el siguiente notebook. No unir todavía EVA con la dimensión; primero comprobar su cardinalidad.

In [4]:
dimension = divipola.rename(columns={"Código Municipio": "codigo_municipio",
    "Código Departamento": "codigo_departamento", "Nombre Municipio": "municipio",
    "Nombre Departamento": "departamento"})[
        ["codigo_municipio", "codigo_departamento", "municipio", "departamento"]].copy()
dimension["codigo_municipio"] = dimension["codigo_municipio"].str.zfill(5)
dimension["codigo_departamento"] = dimension["codigo_departamento"].str.zfill(2)
assert dimension["codigo_municipio"].notna().all()
assert dimension["codigo_municipio"].str.fullmatch(r"\d{5}").all()
assert dimension["codigo_municipio"].is_unique
archivo_eva = OUT / "eva.parquet"
eva.to_parquet(archivo_eva, index=False)
dimension.to_parquet(OUT / "dim_municipio.parquet", index=False)
pd.testing.assert_frame_equal(eva, pd.read_parquet(archivo_eva))
pd.testing.assert_frame_equal(dimension, pd.read_parquet(OUT / "dim_municipio.parquet"))
print("Parquet verificado:", archivo_eva)

Parquet verificado: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/eva.parquet


## 4. Leer con Spark: transformaciones y acciones
`read.parquet` obtiene el esquema; `filter` y `select` construyen un plan. `show`, `count`, escritura y `collect` disparan trabajo. No usar `toPandas()` sobre todos los datos sin evaluar memoria. Aquí solo se visualizan cinco filas.

In [5]:
eva_spark = spark.read.parquet(str(archivo_eva))
eva_spark.printSchema()
aptos = eva_spark.filter(F.col("apto_rendimiento"))
aptos.select("codigo_municipio", "cultivo", "anio", "area_ha", "produccion_t").show(5, truncate=False)
print("Particiones de entrada:", eva_spark.rdd.getNumPartitions())
assert aptos.count() == 159616

root
 |-- codigo_municipio: string (nullable = true)
 |-- codigo_departamento: string (nullable = true)
 |-- anio: long (nullable = true)
 |-- cultivo: string (nullable = true)
 |-- periodo: string (nullable = true)
 |-- estado_fisico: string (nullable = true)
 |-- area_ha: double (nullable = true)
 |-- produccion_t: double (nullable = true)
 |-- rendimiento_publicado: double (nullable = true)
 |-- fila_fuente: long (nullable = true)
 |-- area_ha_texto: string (nullable = true)
 |-- produccion_t_texto: string (nullable = true)
 |-- rendimiento_publicado_texto: string (nullable = true)
 |-- anio_texto: string (nullable = true)
 |-- apto_rendimiento: boolean (nullable = true)



+----------------+-------+----+-------+------------+
|codigo_municipio|cultivo|anio|area_ha|produccion_t|
+----------------+-------+----+-------+------------+
|05001           |Apio   |2021|8.0    |128.0       |
|05001           |Apio   |2021|8.0    |152.0       |
|05001           |Apio   |2022|8.0    |127.44      |
|05001           |Apio   |2022|8.0    |152.0       |
|05001           |Apio   |2023|7.0    |133.0       |
+----------------+-------+----+-------+------------+
only showing top 5 rows
Particiones de entrada: 1


## 5. Agregar sin ejecutar todavía
La unidad es municipio–cultivo–estado físico–año. Calcular suma de producción / suma de área. En `explain` localizar lectura, filtro, agregaciones y `Exchange` (redistribución o shuffle). Explicar por qué cuatro particiones de shuffle no equivalen a cuatro máquinas.

In [6]:
claves = ["codigo_municipio", "cultivo", "estado_fisico", "anio"]
rendimiento = (aptos.groupBy(*claves)
    .agg(F.sum("produccion_t").alias("produccion_t"), F.sum("area_ha").alias("area_ha"))
    .withColumn("rendimiento_t_ha", F.col("produccion_t") / F.col("area_ha")))
rendimiento.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- HashAggregate (6)
      +- Exchange (5)
         +- HashAggregate (4)
            +- Project (3)
               +- Filter (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [codigo_municipio#0, anio#2L, cultivo#3, estado_fisico#5, area_ha#6, produccion_t#7, apto_rendimiento#14]
Batched: true
Location: InMemoryFileIndex [file:/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/eva.parquet]
PushedFilters: [IsNotNull(apto_rendimiento), EqualTo(apto_rendimiento,true)]
ReadSchema: struct<codigo_municipio:string,anio:bigint,cultivo:string,estado_fisico:string,area_ha:double,produccion_t:double,apto_rendimiento:boolean>

(2) Filter
Input [7]: [codigo_municipio#0, anio#2L, cultivo#3, estado_fisico#5, area_ha#6, produccion_t#7, apto_rendimiento#14]
Condition : (isnotnull(apto_rendimiento#14) AND apto_rendimiento#14)

(3) Project
Output [6]: [codigo_municipio#0, anio#2L, cultivo#3, estado_fisico#5

## 6. Medir una acción completa
La siguiente celda incluye lectura del Parquet, filtro, agregación y escritura del resultado. La sesión Spark ya está iniciada. No mide solo la construcción del plan ni incluye su arranque. Las celdas de verificación y visualización se ejecutan aparte.

In [7]:
%%time
rendimiento.write.mode("overwrite").parquet(str(OUT / "spark_rendimiento"))

CPU times: user 1.77 ms, sys: 6.9 ms, total: 8.67 ms
Wall time: 5.09 s


In [8]:
rendimiento_guardado = spark.read.parquet(str(OUT / "spark_rendimiento"))
assert rendimiento_guardado.count() == 107620
rendimiento_guardado.orderBy(*claves).show(5, truncate=False)
archivos = sorted((OUT / "spark_rendimiento").glob("part-*.parquet"))
print("Archivos de datos:", len(archivos))
# La carpeta completa es el dataset; no elegir solo el primer part-*.

+----------------+--------+-------------+----+------------+-------+-----------------+
|codigo_municipio|cultivo |estado_fisico|anio|produccion_t|area_ha|rendimiento_t_ha |
+----------------+--------+-------------+----+------------+-------+-----------------+
|05001           |Aguacate|En fresco    |2019|138.0       |23.0   |6.0              |
|05001           |Aguacate|En fresco    |2020|114.0       |19.0   |6.0              |
|05001           |Aguacate|En fresco    |2021|117.12      |19.52  |6.0              |
|05001           |Aguacate|En fresco    |2022|141.12      |23.52  |6.0              |
|05001           |Aguacate|En fresco    |2023|128.12      |19.52  |6.563524590163935|
+----------------+--------+-------------+----+------------+-------+-----------------+
only showing top 5 rows
Archivos de datos: 2


## 7. Consulta equivalente con DuckDB
Usar el mismo Parquet, predicado y claves. El tiempo incluye lectura, agregación y escritura; no la preparación pandas. DuckDB escribe un archivo y Spark un dataset: reportar esta diferencia y no sacar conclusiones de escalabilidad de una ejecución local.

In [9]:
con = duckdb.connect()
con.execute("SET threads=2")
con.read_parquet(str(archivo_eva)).create_view("eva")
sql = 'SELECT codigo_municipio, cultivo, estado_fisico, anio,\n       SUM(produccion_t) AS produccion_t, SUM(area_ha) AS area_ha,\n       SUM(produccion_t) / SUM(area_ha) AS rendimiento_t_ha\nFROM eva WHERE apto_rendimiento\nGROUP BY codigo_municipio, cultivo, estado_fisico, anio'

In [10]:
%%time
referencia = con.sql(sql).df()
referencia.to_parquet(OUT / "rendimiento_duckdb.parquet", index=False)

CPU times: user 137 ms, sys: 33.1 ms, total: 170 ms
Wall time: 467 ms


## 8. Comparar cada grupo, no solo el total
Este resultado agregado cabe en memoria local. Comprobar claves únicas, cobertura idéntica y las tres medidas numéricas con tolerancias explícitas. Ordenar elimina diferencias de orden entre motores; no redondear antes de comparar.

In [11]:
resultado = pd.read_parquet(OUT / "spark_rendimiento")
assert not referencia.duplicated(claves).any() and not resultado.duplicated(claves).any()
cobertura = referencia[claves].merge(resultado[claves], on=claves, how="outer", validate="one_to_one", indicator=True)
assert cobertura["_merge"].eq("both").all()
columnas = claves + ["produccion_t", "area_ha", "rendimiento_t_ha"]
a = referencia[columnas].sort_values(claves).reset_index(drop=True)
b = resultado[columnas].sort_values(claves).reset_index(drop=True)
pd.testing.assert_frame_equal(a[claves], b[claves], check_dtype=False)
import numpy as np
diferencias = {}
for campo in ["produccion_t", "area_ha", "rendimiento_t_ha"]:
    distintos = ~np.isclose(a[campo], b[campo], atol=0.000001, rtol=0, equal_nan=False)
    diferencias[campo] = int(distintos.sum())
assert not any(diferencias.values()), diferencias
control = {"filas_eva": len(eva), "aptos_rendimiento": int(eva["apto_rendimiento"].sum()),
    "grupos": len(resultado), "diferencias_duckdb": diferencias, "tolerancia_absoluta": 0.000001,
    "spark": spark.version, "duckdb": duckdb.__version__, "master": spark.sparkContext.master,
    "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions")}
with open(OUT / "control_spark.json", "w", encoding="utf-8") as archivo:
    json.dump(control, archivo, ensure_ascii=False, indent=2)
display(control)
con.close()

{'filas_eva': 166732,
 'aptos_rendimiento': 159616,
 'grupos': 107620,
 'diferencias_duckdb': {'produccion_t': 0,
  'area_ha': 0,
  'rendimiento_t_ha': 0},
 'tolerancia_absoluta': 1e-06,
 'spark': '4.0.1',
 'duckdb': '1.4.1',
 'master': 'local[2]',
 'shuffle_partitions': '4'}

## 9. Evidencia y continuación
Guardar el notebook ejecutado con el plan y explicar dónde se lee, filtra, redistribuye y agrega. Entregar el control de equivalencia y dos observaciones sobre el coste de ejecución local. Los controles numéricos corresponden al corte docente, no a todas las versiones futuras de EVA.

Continuar con [06 Joins y ventanas](06_Spark_joins_y_ventanas.ipynb). Detener la sesión antes de abrirlo; su primera celda crea una sesión nueva. Para repetir este notebook, iniciar desde la configuración.

In [12]:
spark.stop()